In [16]:
# General imports
import anthropic
from dotenv import load_dotenv

load_dotenv()

True

In [17]:
from anthropic.types import Message


def add_user_messages(messages, message):
    messages.append(
        {
            "role": "user",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def add_assistant_messages(messages, message):
    messages.append(
        {
            "role": "assistant",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def chat_stream(messages, stop_sequences=[], system=None, tools=None):
    client = anthropic.Anthropic()
    parameters = {
        "model": "claude-sonnet-4-6",
        "max_tokens": 1024 * 10,
        "messages": messages,
        "stop_sequences": stop_sequences,
        "thinking": {"type": "enabled", "budget_tokens": 1024},
    }

    if tools:
        parameters["tools"] = tools
    if system:
        parameters["system"] = system

    return client.messages.stream(**parameters)

In [18]:
messages = []
first_message = "Write one sentence."
print(first_message)
add_user_messages(messages, first_message)

with chat_stream(messages) as stream:
    for event in stream:
        print(event)
        if event.type == "content_block_start":
            if event.content_block.type == "thinking":
                print("\n[Thinking...]", flush=True)
            elif event.content_block.type == "text":
                print("\n[Response:]", flush=True)
        elif event.type == "content_block_delta":
            if event.delta.type == "thinking_delta":
                print(event.delta.thinking, end="", flush=True)
            elif event.delta.type == "text_delta":
                print(event.delta.text, end="", flush=True)

    response = stream.get_final_message()
    print("\n", response)
    add_assistant_messages(messages, response)

Write one sentence.
RawMessageStartEvent(message=Message(id='msg_013MVuvZTfQEBLA3rET5MJNd', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=40, output_tokens=3, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=ThinkingBlock(signature='', thinking='', type='thinking'), index=0, type='content_block_start')

[Thinking...]
RawContentBlockDeltaEvent(delta=ThinkingDelta(thinking='The user wants', type='thinking_delta'), index=0, type='content_block_delta')
The user wantsThinkingEvent(type='thinking', thinking='The user wants', snapshot='The user wants')
RawContentBlockDeltaEvent(delta=ThinkingDelta(thinking=' me to 